# TSA Chapter 12: Spectral Analysis of US Real GDP

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/TSA/blob/main/TSA_ch12/TSA_ch12_gdp/TSA_ch12_gdp.ipynb)

In [ ]:
!pip install matplotlib numpy scipy statsmodels pywt -q

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy.signal import periodogram
from scipy.signal.windows import dpss

In [ ]:
# Style configuration
COLORS = {
    'blue':   '#1A3A6E',
    'red':    '#DC3545',
    'green':  '#2E7D32',
    'orange': '#E67E22',
    'gray':   '#666666',
    'purple': '#8E44AD',
}

plt.rcParams.update({
    'axes.facecolor':     'none',
    'figure.facecolor':   'none',
    'savefig.transparent': True,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'axes.grid':          False,
    'font.size':          9,
    'axes.labelsize':     9,
    'axes.titlesize':     10,
    'legend.fontsize':    8,
    'xtick.labelsize':    8,
    'ytick.labelsize':    8,
})

In [ ]:
# Chart: ch12_gdp_periodogram
# Multitaper spectral estimate of US real GDP with business cycle band
data = sm.datasets.macrodata.load_pandas().data
gdp = np.log(data['realgdp'].values)
T = len(gdp)

tapers, _ = dpss(T, NW=4, Kmax=7, return_ratios=True)
mt_psds = []
for tap in tapers:
    ft = np.fft.rfft(gdp * tap)
    mt_psds.append(np.abs(ft)**2 / T)
p_mt = np.mean(mt_psds, axis=0)
f_mt = np.fft.rfftfreq(T, d=1)

f_raw, p_raw = periodogram(gdp)

fig, ax = plt.subplots(figsize=(10, 4))
ax.semilogy(f_raw[1:], p_raw[1:], color=COLORS['gray'], lw=0.8, alpha=0.5, label='Raw periodogram')
ax.semilogy(f_mt[1:], p_mt[1:], color=COLORS['blue'], lw=2, label='Multitaper (NW=4, K=7)')
ax.axvspan(1/32, 1/6, alpha=0.15, color=COLORS['orange'], label='Business cycle (6–32q)')
ax.set_xlabel('Frequency (cycles/quarter)')
ax.set_ylabel('PSD (log)')
ax.set_title('Multitaper Spectral Estimate of US Real GDP')
fig.patch.set_alpha(0); ax.set_facecolor('none')
ax.spines[['top','right']].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
plt.tight_layout()
plt.savefig('ch12_gdp_periodogram.pdf', bbox_inches='tight', dpi=150)
plt.savefig('ch12_gdp_periodogram.png', bbox_inches='tight', dpi=150)
plt.show()